<a href="https://colab.research.google.com/github/valeriavasquezv/valeriavasquez-portfolio/blob/main/Telecom2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Telecom 2


In [1]:
import pandas as pd

In [2]:
datos = pd.read_csv('https://raw.githubusercontent.com/valeriavasquezv/valeriavasquez-portfolio/refs/heads/main/telecomx_limpio.csv')

In [3]:
datos.tail()

,customerID,Churn,customer,phone,internet,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total
7038,9987-LUTYD,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...",One year,No,Mailed check,55.15,742.90
7039,9992-RRAMN,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'Fiber optic', 'OnlineSecu...",Month-to-month,Yes,Electronic check,85.10,1873.70
7040,9992-UJOEL,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...",Month-to-month,Yes,Mailed check,50.30,92.75
7041,9993-LHIEB,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...",Two year,No,Mailed check,67.85,4627.65
7042,9995-HOTOH,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'No', 'MultipleLines': 'No ph...","{'InternetService': 'DSL', 'OnlineSecurity': '...",Two year,No,Electronic check,59.00,3707.60


In [4]:
import ast

# Convertir de string a diccionario real
datos["customer"] = datos["customer"].apply(ast.literal_eval)
datos["phone"] = datos["phone"].apply(ast.literal_eval)
datos["internet"] = datos["internet"].apply(ast.literal_eval)

In [5]:
customer_expandido = pd.json_normalize(datos["customer"])
phone_expandido = pd.json_normalize(datos["phone"])
internet_expandido = pd.json_normalize(datos["internet"])

datos = pd.concat([
    datos.drop(columns=["customer", "phone", "internet"]),
    customer_expandido, phone_expandido, internet_expandido], axis=1)

datos.shape

(7043, 21)

In [6]:
datos = datos.drop(columns=["customerID"])

In [7]:
# Variable Objetivo

X = datos.drop("Churn", axis=1)
y = datos["Churn"]

In [8]:
# id de Variables categóricas

X.dtypes

,0
Contract,object
PaperlessBilling,object
PaymentMethod,object
Charges.Monthly,float64
Charges.Total,float64
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64


In [9]:
X = datos.drop("Churn", axis=1)
y = datos["Churn"]

X_encoded = pd.get_dummies(X, drop_first=True)

X_encoded.shape

(7043, 30)

In [10]:
datos["Churn"].value_counts(normalize=True)

,proportion
Churn,
No,0.73463
Yes,0.26537


In [11]:
X = X_encoded
y = datos["Churn"]

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

X_train.shape, X_test.shape

((4930, 30), (2113, 30))

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
datos_corr = X_encoded.copy()

# Verificamos qué contiene y
print(type(y))
print(y.head())

# Agregamos la variable objetivo
datos_corr["Churn"] = y.values

# Verificamos que sí se agregó
print(datos_corr.columns)

<class 'pandas.core.series.Series'>
0     No
1     No
2    Yes
3    Yes
4    Yes
Name: Churn, dtype: object
Index(['Charges.Monthly', 'Charges.Total', 'SeniorCitizen', 'tenure',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Churn'],
      dtype='object')


In [22]:
# Convertir Churn a numérico
datos_corr["Churn"] = datos_corr["Churn"].map({"No": 0, "Yes": 1})

# Seleccionar columnas numéricas
datos_corr = datos_corr.select_dtypes(include=["number", "bool"])

# Rellenar nulos por seguridad
datos_corr = datos_corr.fillna(0)

# Calcular correlación
corr_matrix = datos_corr.corr()

corr_churn = corr_matrix["Churn"].sort_values(ascending=False)

corr_churn

,Churn
Churn,1.000000
InternetService_Fiber optic,0.308020
PaymentMethod_Electronic check,0.301919
Charges.Monthly,0.193356
PaperlessBilling_Yes,0.191825
SeniorCitizen,0.150889
StreamingTV_Yes,0.063228
StreamingMovies_Yes,0.061382
MultipleLines_Yes,0.040102
PhoneService_Yes,0.011942
